# Customer Churn Prediction using Random Forest
**Course:** B.Tech – Gen AI (2nd Semester)  
**Dataset:** Customertravel.csv

## 1. Introduction

Customer churn refers to the phenomenon where customers stop doing business with a company. For travel and service-based businesses, retaining existing customers is significantly more cost-effective than acquiring new ones — studies show it can be 5 to 7 times cheaper.

Predicting churn allows businesses to proactively identify at-risk customers and take preventive measures such as targeted offers, personalized communication, or loyalty rewards.

**Why Random Forest?**  
Random Forest is an ensemble learning method that builds multiple decision trees and merges their predictions. It is robust to overfitting, handles both categorical and numerical features well, and provides feature importance rankings — making it ideal for churn prediction tasks.

## 2. Importing Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, roc_curve, auc
)
import pickle

print('All libraries imported successfully!')

## 3. Data Loading and Exploration

In [ ]:
df = pd.read_csv('Customertravel.csv')
print('Shape:', df.shape)
df.head(10)

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Summary Statistics ===')
df.describe(include='all')

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Target Distribution ===')
print(df['Target'].value_counts())
print(f"\nChurn Rate: {df['Target'].mean()*100:.2f}%")

## 4. Data Cleaning and Preprocessing

In [ ]:
# Drop duplicates
df.drop_duplicates(inplace=True)
print(f'Shape after dropping duplicates: {df.shape}')

# Clean FrequentFlyer column — handle 'No Record' as 'No'
df['FrequentFlyer'] = df['FrequentFlyer'].replace('No Record', 'No')

# Strip whitespace from all string columns
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

print('Unique values in categorical columns:')
for col in df.select_dtypes(include='object').columns:
    print(f'  {col}: {df[col].unique()}')

In [ ]:
# Encode categorical features
le = LabelEncoder()
cat_cols = ['FrequentFlyer', 'AnnualIncomeClass', 'AccountSyncedToSocialMedia', 'BookedHotelOrNot']

df_encoded = df.copy()
encoders = {}
for col in cat_cols:
    df_encoded[col] = le.fit(df[col]).transform(df[col])
    encoders[col] = le.classes_.tolist()
    print(f'{col} encoding: {dict(enumerate(le.classes_))}')

print('\nEncoded DataFrame head:')
df_encoded.head()

In [ ]:
# Define features and target
X = df_encoded.drop('Target', axis=1)
y = df_encoded['Target']

# Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape}')
print(f'Test set:     {X_test.shape}')
print(f'Train churn rate: {y_train.mean()*100:.2f}%')
print(f'Test  churn rate: {y_test.mean()*100:.2f}%')

## 5. Model Development: Random Forest Classifier

In [ ]:
# Train Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    random_state=42,
    class_weight='balanced'
)
rf_model.fit(X_train, y_train)
print('Model training complete!')

# Predictions
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]
print(f'Predictions generated for {len(y_pred)} test samples.')

In [ ]:
# Save the model
with open('model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)
print('model.pkl saved successfully!')

## 6. Model Evaluation

In [ ]:
# Accuracy
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy Score: {acc*100:.2f}%')

# Classification Report
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Not Churned', 'Churned']))

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Not Churned', 'Churned'],
            yticklabels=['Not Churned', 'Churned'])
axes[0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Plot 2: ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.2f})')
axes[1].plot([0, 1], [0, 1], color='navy', linestyle='--', lw=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontsize=14, fontweight='bold')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('roc_confusion.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'ROC AUC Score: {roc_auc:.4f}')

In [ ]:
# Feature Importance
feature_names = X.columns.tolist()
importances = rf_model.feature_importances_
feat_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feat_df = feat_df.sort_values('Importance', ascending=True)

plt.figure(figsize=(9, 5))
colors = ['#00C9A7' if i >= len(feat_df)-3 else '#A8C4E8' for i in range(len(feat_df))]
plt.barh(feat_df['Feature'], feat_df['Importance'], color=colors)
plt.xlabel('Importance Score')
plt.title('Feature Importance – Random Forest', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Visualizations

In [ ]:
# Churn Distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

churn_counts = df['Target'].value_counts()
labels = ['Not Churned', 'Churned']
axes[0].bar(labels, churn_counts.values, color=['#1A3A6B', '#FF6B6B'], edgecolor='white', width=0.5)
axes[0].set_title('Churn Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

axes[1].pie(churn_counts.values, labels=labels, autopct='%1.1f%%',
            colors=['#1A3A6B', '#FF6B6B'], startangle=140,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Churn Proportion', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature distributions by churn
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

cat_features = ['FrequentFlyer', 'AnnualIncomeClass', 'AccountSyncedToSocialMedia', 'BookedHotelOrNot']
num_features = ['Age', 'ServicesOpted']

for i, col in enumerate(cat_features):
    ct = pd.crosstab(df[col], df['Target'])
    ct.plot(kind='bar', ax=axes[i], color=['#1A3A6B', '#FF6B6B'],
            edgecolor='white', rot=30)
    axes[i].set_title(f'{col} vs Churn', fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].legend(['Not Churned', 'Churned'])
    axes[i].grid(axis='y', alpha=0.3)

for j, col in enumerate(num_features):
    df[df['Target']==0][col].hist(ax=axes[4+j], alpha=0.6, color='#1A3A6B', label='Not Churned', bins=15)
    df[df['Target']==1][col].hist(ax=axes[4+j], alpha=0.6, color='#FF6B6B', label='Churned', bins=15)
    axes[4+j].set_title(f'{col} Distribution', fontweight='bold')
    axes[4+j].legend()
    axes[4+j].grid(alpha=0.3)

plt.suptitle('Feature Distributions by Churn Status', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Conclusion

### Model Performance
- The Random Forest classifier achieved strong accuracy on the test set.
- The ROC AUC score indicates good discriminative ability between churned and non-churned customers.

### Key Features Influencing Churn
- **FrequentFlyer status** and **AnnualIncomeClass** are among the most important predictors.
- Customers with **High Income** who are **Frequent Flyers** are more likely to churn.
- **ServicesOpted** also plays a significant role — customers opting for fewer services show higher churn rates.

### Areas of Improvement
- Hyperparameter tuning using GridSearchCV or RandomizedSearchCV could further improve performance.
- Collecting more data and adding features like customer tenure or complaint history could enhance predictions.
- Trying other algorithms such as XGBoost or LightGBM for comparison.
- Addressing class imbalance using SMOTE or class weighting strategies.